In [27]:
#TEST PURPOSE
# A single dual-key client (keypair_path + evm_private_key_path) that can:

# Pay with MON (EVM) → confirm → provision → add operator → upload/attest — all with the same client instance
# Pay with SOL (Ed25519) → confirm → provision → add operator → upload/attest — all with the same client instance
# Operate on BOTH EVM-paid and SOL-paid lockers interchangeably, without creating separate client instances

In [2]:
from pynukez import Nukez

In [17]:
# By default (auto_bind_operator=True), when a dual-key client confirms an EVM payment, 
# confirm_storage auto-binds the client's Ed25519 pubkey as an operator on the new locker — 
# Re-adding the same pubkey via add_operator afterward will return 409.

#Note on the EVM key path: the below cell uses /Users/zhanson/.keys/treasuries/gateway/staging/evm_key.json 
#but the payment is made with /Users/zhanson/.keys/delegators/evm_key.json (a dual-key constructor). 
#The owner on this locker is the delegators EVM key — 
#you must use that one, not the staging gateway key, or the gateway will reject the signature as not-the-owner.


evm_client = Nukez(
    keypair_path="/Users/zhanson/.keys/delegators/svm_key.json", 
    evm_private_key_path="/Users/zhanson/.keys/delegators/evm_key.json",
    evm_rpc_url="https://monad-mainnet.g.alchemy.com/v2/6DV2Ss9wfT2ekmv6V1uTE"
)

In [4]:
request = evm_client.request_storage(
    units=1,
    provider="gcs",
    pay_network="eip155:143",   # Monad mainnet
    pay_asset="MON",
)

In [5]:
# Sanity-check the override fired:
print("network:        ", request.network)        # expect: monad-mainnet
print("pay_to_address: ", request.pay_to_address) # expect: 0x744C9B35F7625A86ba75d2B62F9D0e257Dd43c2a
print("pay_asset:      ", request.pay_asset)      # expect: MON
print("amount_raw:     ", request.amount_raw)     # expect: 1436781609195402298850

network:         monad-mainnet
pay_to_address:  0x744C9B35F7625A86ba75d2B62F9D0e257Dd43c2a
pay_asset:       MON
amount_raw:      579038796000000000000


In [6]:
# Pick the MON payment option from the quote
mon_opt = next(o for o in request.payment_options if o["pay_asset"] == "MON")

transfer = evm_client.evm_transfer(
    to_address=mon_opt["pay_to_address"],
    amount_raw=int(mon_opt["amount"]),
    pay_asset="MON",
    token_address=None,
    network="monad-mainnet",
)

print(transfer)

TransferResult(signature='0x3af0ed9739a284599f0675bbe75488aaca99820f4a3fa33fc452171d8bbd3a53', to_address='0x744C9B35F7625A86ba75d2B62F9D0e257Dd43c2a', amount_sol=0.0, network='monad-mainnet', chain='evm', pay_asset='MON', amount_raw=579038796000000000000, tx_hash='0x3af0ed9739a284599f0675bbe75488aaca99820f4a3fa33fc452171d8bbd3a53')


In [7]:
receipt2 = evm_client.confirm_storage(
    request.pay_req_id,
    transfer.tx_hash,
    payment_chain="monad-mainnet",
    payment_asset="MON",
)
print(receipt2)

/var/folders/q3/xpnvz_1s07b49gw6xgfbskn80000gn/T/ipykernel_36894/616278822.py:1: DeprecationWarning: Auto-binding Ed25519 operator for EVM payments is deprecated. EVM owners can now operate directly with secp256k1 envelopes. Pass operator_pubkey explicitly or set auto_bind_operator=False. This behavior will be removed in pynukez 4.0.
  receipt2 = evm_client.confirm_storage(


Receipt(id='8573c83375ebf3ac', units=1, payer_pubkey='0xc12e3657ce2ede7fae1d6f5a83b386f6a630fd18', network='eip155:143', created_at=None, provider='gcs', pay_asset='MON', tx_hash='0x3af0ed9739a284599f0675bbe75488aaca99820f4a3fa33fc452171d8bbd3a53', paid_amount='579.038796', paid_raw=579038796000000000000, block_number=68119038, slot=None, sig_alg='secp256k1', unit_price_usd=20.0, price_usd=20.0, authorized_operator='BhBeSkwKyqysZstzkqdf4qAcYfS9r27wEMmouvSVfp1U')


In [8]:
manifest2 = evm_client.provision_locker(receipt2.id)
print(manifest2)

NukezManifest(locker_id='locker_98e0fd7c8593', receipt_id='8573c83375ebf3ac', bucket='nukez', path_prefix='lockers/locker_98e0fd7c8593/', tags=[], cap_token=None, cap_expires_in_sec=None, created_at=None)


In [ ]:
# Owner-only ops on an EVM-paid locker must be signed with the EVM key.
# With the bind_receipt fix in pynukez, the dual-key client auto-selects
# the correct signer (secp256k1 for EVM receipts, ed25519 for SOL receipts)
# as long as per-receipt state is primed. confirm_storage() primes it
# automatically in the same process; across kernel restarts or client
# reconstruction, call evm_client.bind_receipt(receipt2) to restore it.
#
# No separate EVM-only client is needed for owner admin ops.


In [18]:
manifest = evm_client.get_manifest(receipt2.id)
print(manifest.get("operators") or manifest.get("operator_ids"))

None


In [22]:
from pprint import pprint
pprint(evm_client.get_manifest(receipt2.id))

{'created_at': 1776193466,
 'file_count': 0,
 'files': [],
 'hashed_file_count': 0,
 'locker_id': 'locker_98e0fd7c8593',
 'schema': 'locker_files_v1',
 'total_bytes': 0,
 'updated_at': 1776193466}


In [ ]:
# bind_receipt primes owner_identity + sig_alg on the dual-key client so
# _require_signer picks the secp256k1 signer for this EVM-paid locker.
# confirm_storage() above already primed state in this same process — this
# call is a no-op here but is the canonical pattern for cross-session flows
# (kernel restart, receipt loaded from DB, etc.).
evm_client.bind_receipt(receipt2)

# Remove the auto-bound SVM operator to verify owner-only admin works
# through the dual-key client directly.
evm_client.remove_operator(receipt2.id, "BhBeSkwKyqysZstzkqdf4qAcYfS9r27wEMmouvSVfp1U")

In [ ]:
evm_client.add_operator(receipt2.id, "BhBeSkwKyqysZstzkqdf4qAcYfS9r27wEMmouvSVfp1U")